<a href="https://colab.research.google.com/github/MiketheEstimator/Skills-ShowCase/blob/main/BookScrapper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Text Cell 1: Section 1.0 – Architecture Overview & Runtime Environment Initialization
System Architecture Overview
This notebook implements an enterprise-grade, state-driven Hunt
→
→
 Acquire
→
→
 Deliver Digital Acquisition Engine. Rather than functioning as a monolithic scraper, this system separates discovery, policy-driven qualification, binary retrieval, and delivery into decoupled subsystems bound together through a durable relational ledger.                                      '''--- USER INTENT / HUNT GOAL
                               │
                               ▼
                    ┌─────────────────────┐
                    │   HUNT CONTROLLER   │  <-- "What should we search next?"
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │   SOURCE ADAPTERS   │  <-- Pure Search / Resolve / Enumerate
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │  DISCOVERY ENGINE   │  <-- Multi-Source Ingest & Canonical De-duplication
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │QUALIFICATION ENGINE │  <-- Policy Gates (Accept / Reject / Prefer)
                    └──────────┬──────────┘
                               │
             ┌─────────────────┴─────────────────┐
             │       DURABLE SYSTEM LEDGER       │  <-- SQLite System of Record (Audit Trail)
             └───────┬───────────────────┬───────┘
                     │                   │
                     ▼                   ▼
           ┌───────────────────┐ ┌───────────────────┐
           │ACQUISITION ENGINE │ │  DELIVERY ENGINE  │
           │ (Stream/Checksum) │ │(Export/Packaging) │
           └───────────────────┘ └───────────────────┘ ---'''

Code Cell 1: Environment Configuration & Directory Hierarchy


In [1]:
import os
import sys
import json
import time
import hashlib
import sqlite3
import logging
import datetime
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple, Generator
from dataclasses import dataclass, asdict, field
from enum import Enum
import urllib.parse
import urllib.request
import urllib.error

# Setup clean, structured logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("AcquisitionEngine")

# Define deterministic workspace directory tree
BASE_DIR = Path("/content/acquisition_engine") if "google.colab" in sys.modules else Path("./acquisition_engine")
DATA_DIR = BASE_DIR / "data"
STAGING_DIR = BASE_DIR / "staging"
DELIVERY_DIR = BASE_DIR / "delivery"
LEDGER_PATH = BASE_DIR / "ledger.db"

for directory in [BASE_DIR, DATA_DIR, STAGING_DIR, DELIVERY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

logger.info(f"Initialized Acquisition Engine Workspace at: {BASE_DIR.resolve()}")

Text Cell 2: Section 2.0 – Universal Domain Models & CandidateArtifact Contract
Universal Data Abstraction
To prevent the engine from becoming a single-purpose scraper, the fundamental unit of work is generalized into a CandidateArtifact.
While Profile 001 specializes in BOOK and RESEARCH_PAPER acquisition (for applied mathematics, computational engineering, and open-source scientific corpora), the domain model supports any digital artifact type (PROJECT, DATASET, SPECIFICATION, CONTRACT_BID, etc.) without schema modifications.
Domain Schema Contracts
ArtifactType: Categorical classifier defining the payload domain.
ArtifactStatus: Core lifecycle stage governing work queues.
AcquisitionStatus: Granular tracking for binary retrieval attempts.
CandidateArtifact: Canonical domain entity decoupled from any single external platform.

In [2]:
class ArtifactType(str, Enum):
    BOOK = "BOOK"
    RESEARCH_PAPER = "RESEARCH_PAPER"
    DATASET = "DATASET"
    SPECIFICATION = "SPECIFICATION"
    PROJECT = "PROJECT"
    CONTRACT_BID = "CONTRACT_BID"

class ArtifactStatus(str, Enum):
    DISCOVERED = "DISCOVERED"
    SKIPPED = "SKIPPED"
    QUALIFIED = "QUALIFIED"
    QUEUED = "QUEUED"
    ACQUIRED = "ACQUIRED"
    DELIVERED = "DELIVERED"
    FAILED = "FAILED"

class AcquisitionStatus(str, Enum):
    PENDING = "PENDING"
    DOWNLOADING = "DOWNLOADING"
    VERIFIED = "VERIFIED"
    FAILED = "FAILED"
    INTEGRITY_ERROR = "INTEGRITY_ERROR"

@dataclass
class CandidateArtifact:
    artifact_key: str                     # Deterministic SHA-256 / canonical slug
    artifact_type: ArtifactType           # BOOK, RESEARCH_PAPER, etc.
    canonical_title: str                  # Normalized title
    creators: List[str]                   # Authors / Organizations
    publication_year: Optional[int]       # Normalized Year
    language: str                         # ISO-639-1 code (e.g. 'en')
    rights_license: str                   # 'open_access', 'public_domain', etc.
    target_format: str                    # 'pdf', 'epub', 'json', etc.
    discovery_url: str                    # URL where artifact was indexed
    acquisition_url: str                  # Direct link to payload / API endpoint
    source_name: str                      # Adapter name that found it
    source_id: str                        # Native identifier at source (e.g., ArXiv ID, Gutenberg ID)
    metadata: Dict[str, Any] = field(default_factory=dict)
    evidence: Dict[str, Any] = field(default_factory=dict)
    discovered_at: str = field(default_factory=lambda: datetime.datetime.utcnow().isoformat())

    @staticmethod
    def generate_key(title: str, creator: str = "", artifact_type: str = "BOOK") -> str:
        """Generates a reproducible, deterministic key based on normalized metadata."""
        norm_title = "".join(ch.lower() for ch in title if ch.isalnum())
        norm_creator = "".join(ch.lower() for ch in creator if ch.isalnum())
        raw_key = f"{artifact_type}:{norm_title}:{norm_creator}"
        return hashlib.sha256(raw_key.encode("utf-8")).hexdigest()[:20]

Text Cell 3: Section 3.0 – Durable Relational Ledger (System of Record)
The Ledger as the Center of Gravity
The Ledger is the single source of truth across all processes. Adapters never write directly to disks or manage pipelines; they report records to the ledger.
Relational Schema Architecture
Instead of an un-normalized flat table, the ledger is structured into normalized, append-friendly tables:
artifacts: Canonical inventory and lifecycle status.
sources: Configured upstream catalogs, endpoints, and rate limits.
discoveries: Multi-provenance tracking. If one artifact is discovered across 3 sources (e.g. Gutenberg, Internet Archive, Open Library), it results in 1 Artifact
→
→
 3 Discoveries
→
→
 1 Acquisition
→
→
 1 Delivery.
acquisitions: Immutable audit trail of download attempts, SHA-256 verification hashes, and payloads.
deliveries: Physical/cloud delivery tracking to destination targets.
runs & events: Telemetry, diagnostics, and orchestration state tracking.

In [3]:
class LedgerDB:
    """Thread-safe SQLite Ledger managing state machine transitions and audit logs."""

    def __init__(self, db_path: Path = LEDGER_PATH):
        self.db_path = db_path
        self._init_schema()

    def _get_connection(self) -> sqlite3.Connection:
        conn = sqlite3.connect(self.db_path, timeout=30.0)
        conn.row_factory = sqlite3.Row
        conn.execute("PRAGMA journal_mode=WAL;")
        conn.execute("PRAGMA foreign_keys=ON;")
        return conn

    def _init_schema(self):
        with self._get_connection() as conn:
            conn.executescript("""
            CREATE TABLE IF NOT EXISTS sources (
                source_id TEXT PRIMARY KEY,
                source_name TEXT NOT NULL,
                adapter_class TEXT NOT NULL,
                rate_limit_per_min INTEGER DEFAULT 60,
                is_active INTEGER DEFAULT 1,
                last_polled_at TEXT
            );

            CREATE TABLE IF NOT EXISTS artifacts (
                artifact_key TEXT PRIMARY KEY,
                artifact_type TEXT NOT NULL,
                canonical_title TEXT NOT NULL,
                status TEXT NOT NULL,
                language TEXT,
                rights_license TEXT,
                target_format TEXT,
                metadata_json TEXT,
                created_at TEXT NOT NULL,
                updated_at TEXT NOT NULL
            );

            CREATE TABLE IF NOT EXISTS discoveries (
                discovery_id INTEGER PRIMARY KEY AUTOINCREMENT,
                artifact_key TEXT NOT NULL,
                source_id TEXT NOT NULL,
                source_record_id TEXT NOT NULL,
                source_url TEXT NOT NULL,
                raw_metadata_json TEXT,
                discovered_at TEXT NOT NULL,
                FOREIGN KEY (artifact_key) REFERENCES artifacts (artifact_key),
                FOREIGN KEY (source_id) REFERENCES sources (source_id),
                UNIQUE(source_id, source_record_id)
            );

            CREATE TABLE IF NOT EXISTS acquisitions (
                acquisition_id INTEGER PRIMARY KEY AUTOINCREMENT,
                artifact_key TEXT NOT NULL,
                acquisition_url TEXT NOT NULL,
                attempt INTEGER DEFAULT 1,
                status TEXT NOT NULL,
                local_path TEXT,
                checksum TEXT,
                byte_size INTEGER,
                error_message TEXT,
                started_at TEXT NOT NULL,
                finished_at TEXT,
                FOREIGN KEY (artifact_key) REFERENCES artifacts (artifact_key)
            );

            CREATE TABLE IF NOT EXISTS deliveries (
                delivery_id INTEGER PRIMARY KEY AUTOINCREMENT,
                artifact_key TEXT NOT NULL,
                destination_type TEXT NOT NULL,
                destination_path TEXT NOT NULL,
                external_id TEXT,
                status TEXT NOT NULL,
                delivered_at TEXT NOT NULL,
                FOREIGN KEY (artifact_key) REFERENCES artifacts (artifact_key)
            );

            CREATE TABLE IF NOT EXISTS runs (
                run_id TEXT PRIMARY KEY,
                profile_name TEXT NOT NULL,
                target_intent TEXT NOT NULL,
                status TEXT NOT NULL,
                started_at TEXT NOT NULL,
                finished_at TEXT,
                stats_json TEXT
            );

            CREATE TABLE IF NOT EXISTS events (
                event_id INTEGER PRIMARY KEY AUTOINCREMENT,
                run_id TEXT,
                artifact_key TEXT,
                event_type TEXT NOT NULL,
                payload_json TEXT,
                timestamp TEXT NOT NULL
            );
            """)

    def register_source(self, source_id: str, name: str, adapter_cls: str, rate_limit: int = 60):
        with self._get_connection() as conn:
            conn.execute("""
            INSERT OR REPLACE INTO sources (source_id, source_name, adapter_class, rate_limit_per_min, is_active)
            VALUES (?, ?, ?, ?, 1)
            """, (source_id, name, adapter_cls, rate_limit))

    def record_discovery(self, artifact: CandidateArtifact) -> Tuple[bool, str]:
        """
        Records an artifact and its provenance.
        Returns: (is_new_artifact, artifact_key)
        """
        now = datetime.datetime.utcnow().isoformat()
        with self._get_connection() as conn:
            # 1. Insert or ignore canonical artifact
            cursor = conn.execute("SELECT artifact_key, status FROM artifacts WHERE artifact_key = ?", (artifact.artifact_key,))
            row = cursor.fetchone()
            is_new = False

            if not row:
                is_new = True
                conn.execute("""
                INSERT INTO artifacts (artifact_key, artifact_type, canonical_title, status, language, rights_license, target_format, metadata_json, created_at, updated_at)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """, (
                    artifact.artifact_key,
                    artifact.artifact_type.value,
                    artifact.canonical_title,
                    ArtifactStatus.DISCOVERED.value,
                    artifact.language,
                    artifact.rights_license,
                    artifact.target_format,
                    json.dumps(artifact.metadata),
                    now,
                    now
                ))

            # 2. Record source-specific discovery event (deduplicated by source_id + source_record_id)
            conn.execute("""
            INSERT OR IGNORE INTO discoveries (artifact_key, source_id, source_record_id, source_url, raw_metadata_json, discovered_at)
            VALUES (?, ?, ?, ?, ?, ?)
            """, (
                artifact.artifact_key,
                artifact.source_name,
                artifact.source_id,
                artifact.discovery_url,
                json.dumps(artifact.evidence),
                now
            ))

            return is_new, artifact.artifact_key

    def update_artifact_status(self, artifact_key: str, status: ArtifactStatus):
        now = datetime.datetime.utcnow().isoformat()
        with self._get_connection() as conn:
            conn.execute("""
            UPDATE artifacts SET status = ?, updated_at = ? WHERE artifact_key = ?
            """, (status.value, now, artifact_key))

    def record_acquisition(self, artifact_key: str, url: str, status: AcquisitionStatus,
                           local_path: Optional[str] = None, checksum: Optional[str] = None,
                           byte_size: Optional[int] = None, error_msg: Optional[str] = None) -> int:
        now = datetime.datetime.utcnow().isoformat()
        with self._get_connection() as conn:
            cursor = conn.execute("""
            INSERT INTO acquisitions (artifact_key, acquisition_url, status, local_path, checksum, byte_size, error_message, started_at, finished_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (artifact_key, url, status.value, local_path, checksum, byte_size, error_msg, now, now))
            return cursor.lastrowid

    def record_delivery(self, artifact_key: str, dest_type: str, dest_path: str, external_id: Optional[str] = None):
        now = datetime.datetime.utcnow().isoformat()
        with self._get_connection() as conn:
            conn.execute("""
            INSERT INTO deliveries (artifact_key, destination_type, destination_path, external_id, status, delivered_at)
            VALUES (?, ?, ?, ?, 'DELIVERED', ?)
            """, (artifact_key, dest_type, dest_path, external_id, now))

    def get_pending_acquisitions(self, limit: int = 10) -> List[sqlite3.Row]:
        with self._get_connection() as conn:
            cursor = conn.execute("""
            SELECT a.*, d.source_url, d.source_id, d.raw_metadata_json
            FROM artifacts a
            JOIN discoveries d ON a.artifact_key = d.artifact_key
            WHERE a.status = ?
            GROUP BY a.artifact_key
            LIMIT ?
            """, (ArtifactStatus.QUEUED.value, limit))
            return cursor.fetchall()

Text Cell 4: Section 4.0 – Source Adapter Protocol & Concrete Adapters
Source Adapter Strict Contract
Adapters act exclusively as translators. They interrogate external protocols (APIs, OAI-PMH feeds, JSON endpoints, catalog indexes) and normalize external records into raw CandidateArtifact objects.
Architectural Law:
Adapters never write to disk.
Adapters never execute downloads or binary extraction.
Adapters never mutate the ledger.
Implemented Adapters:
ArxivSourceAdapter: Interrogates the real arXiv API (covering Applied Math math.NA, math.PR, Computer Science cs.AI, cs.DS, cs.LG).
GutendexSourceAdapter: Interrogates Project Gutenberg’s global digital index for open-domain mathematical and engineering literature.


In [4]:
class BaseSourceAdapter:
    """Abstract Interface defining the Source Adapter contract."""
    source_name: str = "BASE_SOURCE"

    def search(self, query: str, limit: int = 25) -> Generator[CandidateArtifact, None, None]:
        raise NotImplementedError

    def resolve(self, source_record_id: str) -> Optional[CandidateArtifact]:
        raise NotImplementedError


class ArxivSourceAdapter(BaseSourceAdapter):
    """Production Adapter for arXiv.org API (Open Access STEM Papers)."""
    source_name: str = "ARXIV"
    BASE_URL: str = "http://export.arxiv.org/api/query"

    def search(self, query: str, limit: int = 10) -> Generator[CandidateArtifact, None, None]:
        import xml.etree.ElementTree as ET

        params = {
            "search_query": f"all:{query}",
            "start": 0,
            "max_results": limit
        }
        encoded_url = f"{self.BASE_URL}?{urllib.parse.urlencode(params)}"
        req = urllib.request.Request(encoded_url, headers={"User-Agent": "MTC3AcquisitionEngine/2.0"})

        try:
            with urllib.request.urlopen(req, timeout=15) as response:
                xml_data = response.read().decode("utf-8")

            root = ET.fromstring(xml_data)
            namespace = {"atom": "http://www.w3.org/2005/Atom"}

            for entry in root.findall("atom:entry", namespace):
                raw_id = entry.find("atom:id", namespace).text.strip()
                arxiv_id = raw_id.split("/abs/")[-1]
                title = entry.find("atom:title", namespace).text.strip().replace("\n", " ")
                summary = entry.find("atom:summary", namespace).text.strip().replace("\n", " ")

                authors = [a.find("atom:name", namespace).text.strip() for a in entry.findall("atom:author", namespace)]
                primary_author = authors[0] if authors else "Unknown"

                pdf_url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"
                published = entry.find("atom:published", namespace).text[:4]
                year = int(published) if published.isdigit() else None

                artifact_key = CandidateArtifact.generate_key(title, primary_author, ArtifactType.RESEARCH_PAPER.value)

                yield CandidateArtifact(
                    artifact_key=artifact_key,
                    artifact_type=ArtifactType.RESEARCH_PAPER,
                    canonical_title=title,
                    creators=authors,
                    publication_year=year,
                    language="en",
                    rights_license="open_access",
                    target_format="pdf",
                    discovery_url=raw_id,
                    acquisition_url=pdf_url,
                    source_name=self.source_name,
                    source_id=arxiv_id,
                    metadata={"abstract": summary, "primary_author": primary_author},
                    evidence={"raw_feed": "arxiv_api", "published": published}
                )
        except Exception as e:
            logger.error(f"Error querying arXiv adapter: {e}")


class GutendexSourceAdapter(BaseSourceAdapter):
    """Production Adapter for Project Gutenberg via Gutendex REST API."""
    source_name: str = "GUTENBERG"
    BASE_URL: str = "https://gutendex.com/books"

    def search(self, query: str, limit: int = 10) -> Generator[CandidateArtifact, None, None]:
        params = {"search": query}
        encoded_url = f"{self.BASE_URL}?{urllib.parse.urlencode(params)}"
        req = urllib.request.Request(encoded_url, headers={"User-Agent": "MTC3AcquisitionEngine/2.0"})

        try:
            with urllib.request.urlopen(req, timeout=15) as response:
                data = json.loads(response.read().decode("utf-8"))

            for item in data.get("results", [])[:limit]:
                title = item.get("title", "Untitled").replace("\n", " ")
                authors = [a.get("name", "Unknown") for a in item.get("authors", [])]
                primary_author = authors[0] if authors else "Unknown"
                book_id = str(item.get("id"))

                # Determine download format (prefer EPUB, fallback to PDF or UTF-8 Plain Text)
                formats = item.get("formats", {})
                acq_url = (
                    formats.get("application/epub+zip") or
                    formats.get("application/pdf") or
                    formats.get("text/plain; charset=utf-8") or
                    formats.get("text/plain")
                )

                if not acq_url:
                    continue

                target_format = "epub" if "epub" in acq_url else ("pdf" if "pdf" in acq_url else "txt")
                languages = item.get("languages", ["en"])
                primary_lang = languages[0] if languages else "en"

                artifact_key = CandidateArtifact.generate_key(title, primary_author, ArtifactType.BOOK.value)

                yield CandidateArtifact(
                    artifact_key=artifact_key,
                    artifact_type=ArtifactType.BOOK,
                    canonical_title=title,
                    creators=authors,
                    publication_year=None,
                    language=primary_lang,
                    rights_license="public_domain",
                    target_format=target_format,
                    discovery_url=f"https://www.gutenberg.org/ebooks/{book_id}",
                    acquisition_url=acq_url,
                    source_name=self.source_name,
                    source_id=book_id,
                    metadata={"download_count": item.get("download_count", 0), "subjects": item.get("subjects", [])},
                    evidence={"gutenberg_id": book_id}
                )
        except Exception as e:
            logger.error(f"Error querying Gutendex adapter: {e}")

Text Cell 5: Section 5.0 – Declarative Qualification Engine (Policy Gates)
The Decoupling of Discovery from Qualification
Finding an object (DISCOVERY: "I found this") is strictly divorced from determining acquisition value (QUALIFICATION: "Does Mike want this in his PhD corpus?").
Policy Specification
The QualificationEngine evaluates declarative policy rules containing:
Accept Gates: Required languages, approved licensing models (open_access, public_domain), and approved payload formats (pdf, epub).
Reject Gates: Banned keywords, unauthorized licensing, missing provenance.
Preference Heuristics: Priority ranking based on format hierarchy and relevance keywords.

In [5]:
@dataclass
class QualificationPolicy:
    """Declarative Policy schema defining qualification criteria."""
    artifact_type: ArtifactType
    allowed_languages: List[str]
    allowed_licenses: List[str]
    allowed_formats: List[str]
    required_keywords: List[str] = field(default_factory=list)
    banned_keywords: List[str] = field(default_factory=list)


class QualificationDecision(str, Enum):
    QUALIFIED = "QUALIFIED"
    REJECTED = "REJECTED"


class QualificationEngine:
    """Evaluates CandidateArtifacts against declarative profile policies."""

    def __init__(self, policy: QualificationPolicy):
        self.policy = policy

    def evaluate(self, candidate: CandidateArtifact) -> Tuple[QualificationDecision, str]:
        # 1. Validate Artifact Type
        if candidate.artifact_type != self.policy.artifact_type:
            return QualificationDecision.REJECTED, f"Mismatched type: {candidate.artifact_type} != {self.policy.artifact_type}"

        # 2. Language Gate
        if candidate.language not in self.policy.allowed_languages:
            return QualificationDecision.REJECTED, f"Unsupported language: {candidate.language}"

        # 3. Rights / License Gate
        if candidate.rights_license not in self.policy.allowed_licenses:
            return QualificationDecision.REJECTED, f"Unauthorized rights profile: {candidate.rights_license}"

        # 4. Format Gate
        if candidate.target_format not in self.policy.allowed_formats:
            return QualificationDecision.REJECTED, f"Disallowed format: {candidate.target_format}"

        # 5. Negative Keyword Filter (Reject Gates)
        text_corpus = f"{candidate.canonical_title} {json.dumps(candidate.metadata)}".lower()
        for banned in self.policy.banned_keywords:
            if banned.lower() in text_corpus:
                return QualificationDecision.REJECTED, f"Triggered banned keyword: '{banned}'"

        # 6. Positive Keyword Gate (if configured)
        if self.policy.required_keywords:
            matched = any(req.lower() in text_corpus for req in self.policy.required_keywords)
            if not matched:
                return QualificationDecision.REJECTED, "Failed required keywords match"

        return QualificationDecision.QUALIFIED, "Passed all qualification gates."

Text Cell 6: Section 6.0 – Discovery Engine & Canonical Deduplication
Multi-Source Ingest Coordination
The DiscoveryEngine coordinates adapters, captures discovered entities, and passes them through the QualificationEngine.
Deduplication Mechanics
Discovered objects are mapped through CandidateArtifact.generate_key().
If an artifact already exists in the ledger from a previous search or separate source, the ledger records a new entry in discoveries (enriching provenance) without creating duplicate acquisition work.

In [6]:
class DiscoveryEngine:
    """Coordinates Source Adapters, executes Qualification, and writes state to the Ledger."""

    def __init__(self, ledger: LedgerDB, qualification_engine: QualificationEngine):
        self.ledger = ledger
        self.qualifier = qualification_engine
        self.adapters: Dict[str, BaseSourceAdapter] = {}

    def register_adapter(self, adapter: BaseSourceAdapter):
        self.adapters[adapter.source_name] = adapter
        self.ledger.register_source(
            source_id=adapter.source_name,
            name=adapter.__class__.__name__,
            adapter_cls=adapter.__class__.__name__
        )

    def discover(self, query: str, limit_per_source: int = 5) -> Dict[str, int]:
        stats = {"discovered": 0, "qualified": 0, "skipped": 0, "duplicates": 0}

        for name, adapter in self.adapters.items():
            logger.info(f"Interrogating Source Adapter: [{name}] for query: '{query}'")
            for candidate in adapter.search(query=query, limit=limit_per_source):
                stats["discovered"] += 1

                # 1. Record discovery to Ledger
                is_new, artifact_key = self.ledger.record_discovery(candidate)
                if not is_new:
                    stats["duplicates"] += 1
                    logger.info(f"Duplicate artifact provenance mapped: {candidate.canonical_title[:40]}...")
                    continue

                # 2. Qualify Candidate
                decision, reason = self.qualifier.evaluate(candidate)
                if decision == QualificationDecision.QUALIFIED:
                    stats["qualified"] += 1
                    self.ledger.update_artifact_status(artifact_key, ArtifactStatus.QUEUED)
                    logger.info(f"QUALIFIED -> QUEUED: [{candidate.target_format.upper()}] {candidate.canonical_title[:50]}")
                else:
                    stats["skipped"] += 1
                    self.ledger.update_artifact_status(artifact_key, ArtifactStatus.SKIPPED)
                    logger.debug(f"SKIPPED: {candidate.canonical_title[:40]} | Reason: {reason}")

        return stats

Text Cell 7: Section 7.0 – Hardened Acquisition Engine (Rate-Limiter, SHA-256 Checksum, Retries)
Reliable Binary Ingest
The AcquisitionEngine is responsible for binary extraction. It operates independently from Discovery, polling the ledger for artifacts in QUEUED status.
Integrity & Verification Safeguards
Network Streaming: Streams HTTP payloads in 64KB chunks directly into the staging directory to prevent out-of-memory errors on large PDFs/archives.
SHA-256 Checksum Calculation: Calculates the exact SHA-256 digest in real-time as bytes stream to disk.
Payload Verification: Verifies minimum byte-size thresholds to detect rate-limit blocks or empty HTML error pages disguised as PDFs.
Retry Loop with Backoff: Automatic retry mechanism updating ledger status to FAILED or ACQUIRED.

In [7]:
class AcquisitionEngine:
    """Pulls QUEUED artifacts from the ledger, downloads binaries, verifies integrity, and updates records."""

    def __init__(self, ledger: LedgerDB, staging_dir: Path = STAGING_DIR, max_retries: int = 3):
        self.ledger = ledger
        self.staging_dir = staging_dir
        self.max_retries = max_retries

    def _stream_download(self, url: str, destination: Path) -> Tuple[str, int]:
        """Streams binary data to disk while computing SHA-256 digest in real-time."""
        sha256 = hashlib.sha256()
        total_bytes = 0

        req = urllib.request.Request(
            url,
            headers={"User-Agent": "MTC3AcquisitionEngine/2.0 (Dual-PhD Corpus Research)"}
        )

        with urllib.request.urlopen(req, timeout=30) as response, open(destination, "wb") as f:
            while chunk := response.read(64 * 1024):
                sha256.update(chunk)
                f.write(chunk)
                total_bytes += len(chunk)

        return sha256.hexdigest(), total_bytes

    def process_queue(self, batch_size: int = 5) -> int:
        queued_items = self.ledger.get_pending_acquisitions(limit=batch_size)
        acquired_count = 0

        for item in queued_items:
            artifact_key = item["artifact_key"]
            target_format = item["target_format"]
            title = item["canonical_title"]

            # Reconstruct metadata
            meta = json.loads(item["metadata_json"]) if item["metadata_json"] else {}
            raw_meta = json.loads(item["raw_metadata_json"]) if item["raw_metadata_json"] else {}

            # Determine direct download URL
            if item["artifact_type"] == ArtifactType.RESEARCH_PAPER.value:
                acq_url = f"https://arxiv.org/pdf/{item['source_id']}.pdf"
            else:
                acq_url = item["source_url"]  # Or resolve from metadata

            staging_file = self.staging_dir / f"{artifact_key}.{target_format}"
            logger.info(f"Acquiring [{artifact_key[:8]}]: {title[:40]}...")

            success = False
            for attempt in range(1, self.max_retries + 1):
                try:
                    time.sleep(1.0)  # Defensive pacing
                    checksum, byte_size = self._stream_download(acq_url, staging_file)

                    # Sanity check: Ensure payload is not an empty error stub (< 1KB)
                    if byte_size < 1024:
                        raise ValueError(f"Acquired payload suspicious: byte size {byte_size} < 1024 bytes")

                    self.ledger.record_acquisition(
                        artifact_key=artifact_key,
                        url=acq_url,
                        status=AcquisitionStatus.VERIFIED,
                        local_path=str(staging_file),
                        checksum=checksum,
                        byte_size=byte_size
                    )
                    self.ledger.update_artifact_status(artifact_key, ArtifactStatus.ACQUIRED)
                    logger.info(f"ACQUIRED: {title[:40]} | Size: {byte_size / 1024:.1f} KB | Hash: {checksum[:8]}...")
                    acquired_count += 1
                    success = True
                    break

                except Exception as e:
                    logger.warning(f"Acquisition Attempt {attempt}/{self.max_retries} failed for {artifact_key[:8]}: {e}")
                    if staging_file.exists():
                        staging_file.unlink()

            if not success:
                logger.error(f"PERMANENT FAILURE: Unable to acquire artifact {artifact_key}")
                self.ledger.record_acquisition(
                    artifact_key=artifact_key,
                    url=acq_url,
                    status=AcquisitionStatus.FAILED,
                    error_msg="Max retries exhausted"
                )
                self.ledger.update_artifact_status(artifact_key, ArtifactStatus.FAILED)

        return acquired_count

Text Cell 8: Section 8.0 – Idempotent Delivery Engine
Safe Physical & Structured Staging
The DeliveryEngine takes verified payloads from the internal staging directory and places them into their permanent structure (e.g., categorized local directories, Google Drive mounts, or institutional repositories).
Companion Metadata Bundling
For every delivered binary, the engine generates an atomic companion metadata JSON record containing:
Canonical metadata and creator list.
Source provenance and URL.
Cryptographic SHA-256 verification signature.
Timestamped delivery audit entry.


In [8]:
class DeliveryEngine:
    """Transfers ACQUIRED artifacts to persistent destination layouts and exports companion metadata."""

    def __init__(self, ledger: LedgerDB, delivery_base: Path = DELIVERY_DIR):
        self.ledger = ledger
        self.delivery_base = delivery_base

    def deliver_acquired(self) -> int:
        with self.ledger._get_connection() as conn:
            cursor = conn.execute("""
            SELECT a.artifact_key, a.artifact_type, a.canonical_title, a.target_format, a.metadata_json,
                   ac.local_path, ac.checksum, ac.byte_size
            FROM artifacts a
            JOIN acquisitions ac ON a.artifact_key = ac.artifact_key
            WHERE a.status = ? AND ac.status = ?
            """, (ArtifactStatus.ACQUIRED.value, AcquisitionStatus.VERIFIED.value))
            ready_items = cursor.fetchall()

        delivered_count = 0
        for item in ready_items:
            key = item["artifact_key"]
            atype = item["artifact_type"]
            src_path = Path(item["local_path"])

            if not src_path.exists():
                logger.error(f"Delivery failed: Source binary missing at {src_path}")
                continue

            # Create category folder (e.g. /delivery/RESEARCH_PAPER/)
            dest_dir = self.delivery_base / atype
            dest_dir.mkdir(parents=True, exist_ok=True)

            dest_binary = dest_dir / f"{key}.{item['target_format']}"
            dest_meta = dest_dir / f"{key}.meta.json"

            # Atomic move / copy
            with open(src_path, "rb") as f_src, open(dest_binary, "wb") as f_dst:
                f_dst.write(f_src.read())

            # Write structured companion metadata
            metadata_payload = {
                "artifact_key": key,
                "title": item["canonical_title"],
                "artifact_type": atype,
                "checksum_sha256": item["checksum"],
                "byte_size": item["byte_size"],
                "delivered_at": datetime.datetime.utcnow().isoformat(),
                "metadata": json.loads(item["metadata_json"]) if item["metadata_json"] else {}
            }
            with open(dest_meta, "w", encoding="utf-8") as f:
                json.dump(metadata_payload, f, indent=2)

            # Update ledger state
            self.ledger.record_delivery(
                artifact_key=key,
                dest_type="LOCAL_STRUCTURED_STORE",
                dest_path=str(dest_binary)
            )
            self.ledger.update_artifact_status(key, ArtifactStatus.DELIVERED)

            # Clean staging
            src_path.unlink(missing_ok=True)
            delivered_count += 1
            logger.info(f"DELIVERED -> [{dest_binary.name}]")

        return delivered_count

Text Cell 9: Section 9.0 – The Hunt Controller (Intent-Driven Search Planning)
Dynamic Search Planning
The HuntController sits above discovery. Its job is not crawling; its job is answering: "What should we search next to fulfill the user's intent?"
Gap Analysis Loop
User Intent: e.g., "Acquire 5 English Applied Mathematics & Distributed Systems foundation papers."
Ledger Inspection: Calculates currently qualified and delivered items against target quotas.
Planning Vector Generation: Generates targeted query vectors for specific source adapters when coverage gaps exist.

In [9]:
@dataclass
class HuntIntent:
    target_profile: str                   # Profile identifier (e.g. 'DUAL_PHD_MATH_CS')
    target_count: int                     # Target number of artifacts
    search_terms: List[str]               # Ordered keyword exploration list
    artifact_type: ArtifactType           # BOOK, RESEARCH_PAPER, etc.


class HuntController:
    """Evaluates ledger coverage against high-level intent and issues directed search directives."""

    def __init__(self, ledger: LedgerDB, intent: HuntIntent):
        self.ledger = ledger
        self.intent = intent

    def get_coverage_delta(self) -> int:
        with self.ledger._get_connection() as conn:
            cursor = conn.execute("""
            SELECT COUNT(*) FROM artifacts
            WHERE artifact_type = ? AND status IN (?, ?)
            """, (self.intent.artifact_type.value, ArtifactStatus.DELIVERED.value, ArtifactStatus.QUEUED.value))
            current_count = cursor.fetchone()[0]

        return max(0, self.intent.target_count - current_count)

    def plan_next_search_vector(self) -> List[str]:
        delta = self.get_coverage_delta()
        if delta <= 0:
            logger.info("Goal reached: No further exploration vectors required.")
            return []

        logger.info(f"Hunt Gap Analysis: {delta} items required to satisfy target.")
        return self.intent.search_terms

Text Cell 10: Section 10.0 – Central Orchestrator & State Machine Controller
Pipeline Orchestration
The PipelineOrchestrator coordinates the entire lifecycle. It does not perform network operations directly; instead, it executes the state machine loop:
Trigger Hunt Controller to evaluate goals and query strategies.
Trigger Discovery Engine across configured Source Adapters.
Commit qualified items into the Ledger.
Trigger Acquisition Engine to download and verify binaries.
Trigger Delivery Engine to stage and export structured assets.
Generate an executive Audit Report.

In [10]:
class PipelineOrchestrator:
    """Master controller scheduling, invoking, and auditing the full Hunt-Acquire-Deliver lifecycle."""

    def __init__(self,
                 ledger: LedgerDB,
                 discovery: DiscoveryEngine,
                 acquisition: AcquisitionEngine,
                 delivery: DeliveryEngine,
                 hunt_controller: HuntController):
        self.ledger = ledger
        self.discovery = discovery
        self.acquisition = acquisition
        self.delivery = delivery
        self.hunt_controller = hunt_controller

    def execute_hunt_cycle(self, limit_per_vector: int = 3):
        run_id = f"RUN_{int(time.time())}"
        logger.info(f"=== INITIATING ACQUISITION CYCLE [{run_id}] ===")

        # 1. Evaluate Hunt Plan
        search_vectors = self.hunt_controller.plan_next_search_vector()
        if not search_vectors:
            logger.info("Pipeline idle: Hunt goals satisfied.")
            return

        # 2. Execute Discovery Phase
        discovery_summary = {"discovered": 0, "qualified": 0, "skipped": 0, "duplicates": 0}
        for query in search_vectors:
            stats = self.discovery.discover(query=query, limit_per_source=limit_per_vector)
            for k in discovery_summary:
                discovery_summary[k] += stats[k]

        logger.info(f"Discovery Phase Completed: {discovery_summary}")

        # 3. Execute Acquisition Phase
        logger.info("Triggering Acquisition Engine Queue...")
        acquired_total = self.acquisition.process_queue(batch_size=10)
        logger.info(f"Acquisition Phase Completed: {acquired_total} items verified.")

        # 4. Execute Delivery Phase
        logger.info("Triggering Delivery Engine...")
        delivered_total = self.delivery.deliver_acquired()
        logger.info(f"Delivery Phase Completed: {delivered_total} items archived.")

        logger.info(f"=== COMPLETED ACQUISITION CYCLE [{run_id}] ===")

    def generate_audit_report(self):
        """Prints a comprehensive tabular audit of the persistent ledger state."""
        with self.ledger._get_connection() as conn:
            cursor = conn.execute("""
            SELECT status, COUNT(*) as count FROM artifacts GROUP BY status
            """)
            status_counts = dict(cursor.fetchall())

            cursor = conn.execute("""
            SELECT a.artifact_key, a.canonical_title, a.status, a.target_format,
                   ac.byte_size, ac.checksum, d.source_id
            FROM artifacts a
            LEFT JOIN acquisitions ac ON a.artifact_key = ac.artifact_key
            LEFT JOIN discoveries d ON a.artifact_key = d.artifact_key
            """)
            rows = cursor.fetchall()

        print("\n" + "="*85)
        print("                  MTC3 ACQUISITION LEDGER AUDIT REPORT")
        print("="*85)
        print(f"Artifact Status Summary: {json.dumps(status_counts, indent=2)}")
        print("-"*85)
        print(f"{'KEY':<10} | {'STATUS':<11} | {'FMT':<4} | {'SIZE (KB)':<10} | {'TITLE'}")
        print("-"*85)
        for r in rows:
            size_kb = f"{r['byte_size']/1024:.1f}" if r['byte_size'] else "N/A"
            title = r['canonical_title'][:42] + "..." if len(r['canonical_title']) > 42 else r['canonical_title']
            print(f"{r['artifact_key'][:10]:<10} | {r['status']:<11} | {r['target_format']:<4} | {size_kb:<10} | {title}")
        print("="*85 + "\n")

Text Cell 11: Section 11.0 – Profile 001 Execution & Ledger Verification Run
Dual PhD Corpus Acquisition Execution
In this final execution cell:
We configure Profile 001 (Applied Mathematics & Distributed Computing Research Corpus).
We initialize the declarative Qualification Policy (accepting pdf and epub, requiring open_access or public_domain, filtering out unwanted topics).
We wire the ArXiv and Gutendex adapters into the DiscoveryEngine.
We define the Hunt Intent (targeting foundational distributed algorithms and numerical linear algebra topics).
The Orchestrator executes the live run against real public APIs, streams verified PDF/EPUB binaries to local storage, saves companion metadata, and prints the persistent ledger audit table.

In [11]:
# 1. Instantiate the Durable Ledger
ledger = LedgerDB(LEDGER_PATH)

# 2. Define Declarative Qualification Policy for Profile 001
stem_phd_policy = QualificationPolicy(
    artifact_type=ArtifactType.RESEARCH_PAPER,
    allowed_languages=["en"],
    allowed_licenses=["open_access", "public_domain"],
    allowed_formats=["pdf", "epub"],
    required_keywords=[],                    # Open search within domain
    banned_keywords=["cryptocurrency", "nft"] # Filter non-academic noise
)
qualification_engine = QualificationEngine(stem_phd_policy)

# 3. Instantiate Engines and Connect Adapters
discovery_engine = DiscoveryEngine(ledger, qualification_engine)
discovery_engine.register_adapter(ArxivSourceAdapter())

acquisition_engine = AcquisitionEngine(ledger, staging_dir=STAGING_DIR)
delivery_engine = DeliveryEngine(ledger, delivery_base=DELIVERY_DIR)

# 4. Configure User Hunt Intent
phd_intent = HuntIntent(
    target_profile="DUAL_PHD_APPLIED_MATH_CS",
    target_count=3,
    search_terms=[
        "distributed consensus numerical methods",
        "convex optimization algorithms"
    ],
    artifact_type=ArtifactType.RESEARCH_PAPER
)

hunt_controller = HuntController(ledger, phd_intent)

# 5. Build and Run the Master Orchestrator
orchestrator = PipelineOrchestrator(
    ledger=ledger,
    discovery=discovery_engine,
    acquisition=acquisition_engine,
    delivery=delivery_engine,
    hunt_controller=hunt_controller
)

# Execute the pipeline
orchestrator.execute_hunt_cycle(limit_per_vector=2)

# 6. Display Ledger System-of-Record Audit Report
orchestrator.generate_audit_report()

# 7. Verify Delivered File System Hierarchy
print("Delivered Physical Artifact Directory Hierarchy:")
for root, dirs, files in os.walk(DELIVERY_DIR):
    level = root.replace(str(DELIVERY_DIR), '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")

/tmp/ipykernel_4164/1039464186.py:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  discovered_at: str = field(default_factory=lambda: datetime.datetime.utcnow().isoformat())
/tmp/ipykernel_4164/3589440521.py:111: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.datetime.utcnow().isoformat()
/tmp/ipykernel_4164/3589440521.py:152: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.datetime.utcnow().isoformat()
ERROR:AcquisitionEngine:PERMANENT FAILURE: Unable to acquire artifact 872c1553815eadf5af


                  MTC3 ACQUISITION LEDGER AUDIT REPORT
Artifact Status Summary: {
  "FAILED": 4
}
-------------------------------------------------------------------------------------
KEY        | STATUS      | FMT  | SIZE (KB)  | TITLE
-------------------------------------------------------------------------------------
98f837a099 | FAILED      | pdf  | N/A        | A Flux Conserving Meshfree Method for Cons...
872c155381 | FAILED      | pdf  | N/A        | Information-Theoretic Privacy in Distribut...
9e33893707 | FAILED      | pdf  | N/A        | Gradient-based Algorithms for Convex Discr...
9cc1cce31a | FAILED      | pdf  | N/A        | Convex Optimization: Algorithms and Comple...

Delivered Physical Artifact Directory Hierarchy:
delivery/


Text Cell 1: Section 1.0 – Architecture Overview & Environment Initialization
Universal Acquisition Pipeline Architecture
This system is an automated, state-driven Universal Digital Acquisition Engine designed to discover, qualify, acquire, verify, deliver, and record digital artifacts into a durable system of record.
code
Code
HUNT INTENT / GOAL
                           │
                           ▼
                ┌─────────────────────┐
                │   HUNT CONTROLLER   │  <-- Evaluates gaps: "What do we search next?"
                └──────────┬──────────┘
                           │
                           ▼
                ┌─────────────────────┐
                │   SOURCE ADAPTERS   │  <-- Pure Search/Resolve (Zero Disk/DB side-effects)
                └──────────┬──────────┘
                           │
                           ▼
                ┌─────────────────────┐
                │  DISCOVERY ENGINE   │  <-- Multi-Source Ingest & Canonical De-duplication
                └──────────┬──────────┘
                           │
                           ▼
                ┌─────────────────────┐
                │QUALIFICATION ENGINE │  <-- Policy Gates (Language, Format, Rights, Keywords)
                └──────────┬──────────┘
                           │
         ┌─────────────────┴─────────────────┐
         │       DURABLE SYSTEM LEDGER       │  <-- SQLite System of Record (Audit Trail)
         └───────┬───────────────────┬───────┘
                 │                   │
                 ▼                   ▼
       ┌───────────────────┐ ┌───────────────────┐
       │ACQUISITION ENGINE │ │  DELIVERY ENGINE  │
       │ (Stream/Checksum) │ │(Export/Packaging) │
       └───────────────────┘ └───────────────────┘
Artifact Lifecycle State Machine
DISCOVERED
⟶
QUALIFIED
⟶
QUEUED
⟶
ACQUIRED
⟶
VERIFIED
⟶
DELIVERED
DISCOVERED⟶QUALIFIED⟶QUEUED⟶ACQUIRED⟶VERIFIED⟶DELIVERED

Branches: DISCOVERED
⟶
SKIPPED
and
QUEUED
⟶
FAILED
Branches: DISCOVERED⟶SKIPPEDandQUEUED⟶FAILED
This initial cell configures timezone-aware logging, builds the isolated filesystem tree, and prepares the workspace.

In [12]:
import os
import sys
import json
import time
import hashlib
import sqlite3
import logging
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple, Generator
from dataclasses import dataclass, asdict, field
from enum import Enum
import urllib.parse
import urllib.request
import urllib.error

# Setup clean, structured logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("AcquisitionEngine")

# Standardized ISO UTC timestamp generator (Python 3.12+ compliant)
def get_utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

# Define deterministic workspace directory tree
BASE_DIR = Path("/content/acquisition_engine") if "google.colab" in sys.modules else Path("./acquisition_engine")
DATA_DIR = BASE_DIR / "data"
STAGING_DIR = BASE_DIR / "staging"
DELIVERY_DIR = BASE_DIR / "delivery"
LEDGER_PATH = BASE_DIR / "ledger.db"

for directory in [BASE_DIR, DATA_DIR, STAGING_DIR, DELIVERY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

logger.info(f"Initialized Acquisition Engine Workspace at: {BASE_DIR.resolve()}")

Text Cell 2: Section 2.0 – Universal Domain Models & CandidateArtifact Contract
Universal Data Abstraction
To keep the engine extensible across different target domains (e.g., BOOK, RESEARCH_PAPER, DATASET, SPECIFICATION, CONTRACT_BID), the core data exchange model is standardized as CandidateArtifact.
Core Properties:
artifact_key: SHA-256 deterministic fingerprint generated from normalized metadata to prevent duplicate acquisitions across differing sources.
acquisition_url: Direct, fully qualified endpoint to retrieve the actual binary payload.
rights_license: Provenance flag (open_access, public_domain, etc.) for policy qualification.

In [13]:
class ArtifactType(str, Enum):
    BOOK = "BOOK"
    RESEARCH_PAPER = "RESEARCH_PAPER"
    DATASET = "DATASET"
    SPECIFICATION = "SPECIFICATION"
    PROJECT = "PROJECT"
    CONTRACT_BID = "CONTRACT_BID"

class ArtifactStatus(str, Enum):
    DISCOVERED = "DISCOVERED"
    SKIPPED = "SKIPPED"
    QUALIFIED = "QUALIFIED"
    QUEUED = "QUEUED"
    ACQUIRED = "ACQUIRED"
    DELIVERED = "DELIVERED"
    FAILED = "FAILED"

class AcquisitionStatus(str, Enum):
    PENDING = "PENDING"
    DOWNLOADING = "DOWNLOADING"
    VERIFIED = "VERIFIED"
    FAILED = "FAILED"
    INTEGRITY_ERROR = "INTEGRITY_ERROR"

@dataclass
class CandidateArtifact:
    artifact_key: str                     # Deterministic SHA-256 / canonical slug
    artifact_type: ArtifactType           # BOOK, RESEARCH_PAPER, etc.
    canonical_title: str                  # Normalized title
    creators: List[str]                   # Authors / Organizations
    publication_year: Optional[int]       # Normalized Year
    language: str                         # ISO-639-1 code (e.g., 'en')
    rights_license: str                   # 'open_access', 'public_domain', etc.
    target_format: str                    # 'pdf', 'epub', 'json', etc.
    discovery_url: str                    # URL where artifact metadata was indexed
    acquisition_url: str                  # Direct link to download binary/payload
    source_name: str                      # Adapter name that found it
    source_id: str                        # Native identifier at source (e.g., ArXiv ID, Gutenberg ID)
    metadata: Dict[str, Any] = field(default_factory=dict)
    evidence: Dict[str, Any] = field(default_factory=dict)
    discovered_at: str = field(default_factory=get_utc_now)

    @staticmethod
    def generate_key(title: str, creator: str = "", artifact_type: str = "BOOK") -> str:
        """Generates a deterministic key based on normalized metadata."""
        norm_title = "".join(ch.lower() for ch in title if ch.isalnum())
        norm_creator = "".join(ch.lower() for ch in creator if ch.isalnum())
        raw_key = f"{artifact_type}:{norm_title}:{norm_creator}"
        return hashlib.sha256(raw_key.encode("utf-8")).hexdigest()[:20]

Text Cell 3: Section 3.0 – Durable Relational Ledger (System of Record)
The Ledger as the Center of Gravity
The SQLite Ledger maintains absolute persistent state across runs. In this updated implementation:
discoveries explicitly stores the resolved acquisition_url.
get_pending_acquisitions pulls the verified acquisition_url directly from the discovery record rather than attempting to guess or reconstruct it.
Multiple source discoveries resolve into a single canonical artifact entry, ensuring 1 Artifact
→
→
 N Discoveries
→
→
 1 Download.

In [14]:
class LedgerDB:
    """Thread-safe SQLite Ledger managing state machine transitions and audit logs."""

    def __init__(self, db_path: Path = LEDGER_PATH):
        self.db_path = db_path
        self._init_schema()

    def _get_connection(self) -> sqlite3.Connection:
        conn = sqlite3.connect(self.db_path, timeout=30.0)
        conn.row_factory = sqlite3.Row
        conn.execute("PRAGMA journal_mode=WAL;")
        conn.execute("PRAGMA foreign_keys=ON;")
        return conn

    def _init_schema(self):
        with self._get_connection() as conn:
            conn.executescript("""
            CREATE TABLE IF NOT EXISTS sources (
                source_id TEXT PRIMARY KEY,
                source_name TEXT NOT NULL,
                adapter_class TEXT NOT NULL,
                rate_limit_per_min INTEGER DEFAULT 60,
                is_active INTEGER DEFAULT 1,
                last_polled_at TEXT
            );

            CREATE TABLE IF NOT EXISTS artifacts (
                artifact_key TEXT PRIMARY KEY,
                artifact_type TEXT NOT NULL,
                canonical_title TEXT NOT NULL,
                status TEXT NOT NULL,
                language TEXT,
                rights_license TEXT,
                target_format TEXT,
                metadata_json TEXT,
                created_at TEXT NOT NULL,
                updated_at TEXT NOT NULL
            );

            CREATE TABLE IF NOT EXISTS discoveries (
                discovery_id INTEGER PRIMARY KEY AUTOINCREMENT,
                artifact_key TEXT NOT NULL,
                source_id TEXT NOT NULL,
                source_record_id TEXT NOT NULL,
                source_url TEXT NOT NULL,
                acquisition_url TEXT NOT NULL,
                raw_metadata_json TEXT,
                discovered_at TEXT NOT NULL,
                FOREIGN KEY (artifact_key) REFERENCES artifacts (artifact_key),
                FOREIGN KEY (source_id) REFERENCES sources (source_id),
                UNIQUE(source_id, source_record_id)
            );

            CREATE TABLE IF NOT EXISTS acquisitions (
                acquisition_id INTEGER PRIMARY KEY AUTOINCREMENT,
                artifact_key TEXT NOT NULL,
                acquisition_url TEXT NOT NULL,
                attempt INTEGER DEFAULT 1,
                status TEXT NOT NULL,
                local_path TEXT,
                checksum TEXT,
                byte_size INTEGER,
                error_message TEXT,
                started_at TEXT NOT NULL,
                finished_at TEXT,
                FOREIGN KEY (artifact_key) REFERENCES artifacts (artifact_key)
            );

            CREATE TABLE IF NOT EXISTS deliveries (
                delivery_id INTEGER PRIMARY KEY AUTOINCREMENT,
                artifact_key TEXT NOT NULL,
                destination_type TEXT NOT NULL,
                destination_path TEXT NOT NULL,
                external_id TEXT,
                status TEXT NOT NULL,
                delivered_at TEXT NOT NULL,
                FOREIGN KEY (artifact_key) REFERENCES artifacts (artifact_key)
            );

            CREATE TABLE IF NOT EXISTS runs (
                run_id TEXT PRIMARY KEY,
                profile_name TEXT NOT NULL,
                target_intent TEXT NOT NULL,
                status TEXT NOT NULL,
                started_at TEXT NOT NULL,
                finished_at TEXT,
                stats_json TEXT
            );

            CREATE TABLE IF NOT EXISTS events (
                event_id INTEGER PRIMARY KEY AUTOINCREMENT,
                run_id TEXT,
                artifact_key TEXT,
                event_type TEXT NOT NULL,
                payload_json TEXT,
                timestamp TEXT NOT NULL
            );
            """)

    def register_source(self, source_id: str, name: str, adapter_cls: str, rate_limit: int = 60):
        with self._get_connection() as conn:
            conn.execute("""
            INSERT OR REPLACE INTO sources (source_id, source_name, adapter_class, rate_limit_per_min, is_active)
            VALUES (?, ?, ?, ?, 1)
            """, (source_id, name, adapter_cls, rate_limit))

    def record_discovery(self, artifact: CandidateArtifact) -> Tuple[bool, str]:
        """
        Records an artifact and its discovery provenance.
        Returns: (is_new_artifact, artifact_key)
        """
        now = get_utc_now()
        with self._get_connection() as conn:
            # 1. Insert or ignore canonical artifact
            cursor = conn.execute("SELECT artifact_key, status FROM artifacts WHERE artifact_key = ?", (artifact.artifact_key,))
            row = cursor.fetchone()
            is_new = False

            if not row:
                is_new = True
                conn.execute("""
                INSERT INTO artifacts (artifact_key, artifact_type, canonical_title, status, language, rights_license, target_format, metadata_json, created_at, updated_at)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """, (
                    artifact.artifact_key,
                    artifact.artifact_type.value,
                    artifact.canonical_title,
                    ArtifactStatus.DISCOVERED.value,
                    artifact.language,
                    artifact.rights_license,
                    artifact.target_format,
                    json.dumps(artifact.metadata),
                    now,
                    now
                ))

            # 2. Record source-specific discovery event (stores verified acquisition_url)
            conn.execute("""
            INSERT OR IGNORE INTO discoveries (artifact_key, source_id, source_record_id, source_url, acquisition_url, raw_metadata_json, discovered_at)
            VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (
                artifact.artifact_key,
                artifact.source_name,
                artifact.source_id,
                artifact.discovery_url,
                artifact.acquisition_url,
                json.dumps(artifact.evidence),
                now
            ))

            return is_new, artifact.artifact_key

    def update_artifact_status(self, artifact_key: str, status: ArtifactStatus):
        now = get_utc_now()
        with self._get_connection() as conn:
            conn.execute("""
            UPDATE artifacts SET status = ?, updated_at = ? WHERE artifact_key = ?
            """, (status.value, now, artifact_key))

    def record_acquisition(self, artifact_key: str, url: str, status: AcquisitionStatus,
                           local_path: Optional[str] = None, checksum: Optional[str] = None,
                           byte_size: Optional[int] = None, error_msg: Optional[str] = None) -> int:
        now = get_utc_now()
        with self._get_connection() as conn:
            cursor = conn.execute("""
            INSERT INTO acquisitions (artifact_key, acquisition_url, status, local_path, checksum, byte_size, error_message, started_at, finished_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (artifact_key, url, status.value, local_path, checksum, byte_size, error_msg, now, now))
            return cursor.lastrowid

    def record_delivery(self, artifact_key: str, dest_type: str, dest_path: str, external_id: Optional[str] = None):
        now = get_utc_now()
        with self._get_connection() as conn:
            conn.execute("""
            INSERT INTO deliveries (artifact_key, destination_type, destination_path, external_id, status, delivered_at)
            VALUES (?, ?, ?, ?, 'DELIVERED', ?)
            """, (artifact_key, dest_type, dest_path, external_id, now))

    def get_pending_acquisitions(self, limit: int = 10) -> List[sqlite3.Row]:
        with self._get_connection() as conn:
            cursor = conn.execute("""
            SELECT a.artifact_key, a.artifact_type, a.canonical_title, a.target_format, a.metadata_json,
                   d.acquisition_url, d.source_id, d.source_name, d.raw_metadata_json
            FROM artifacts a
            JOIN (
                SELECT artifact_key, acquisition_url, source_id, source_id as source_name, raw_metadata_json
                FROM discoveries
                GROUP BY artifact_key
            ) d ON a.artifact_key = d.artifact_key
            WHERE a.status = ?
            LIMIT ?
            """, (ArtifactStatus.QUEUED.value, limit))
            return cursor.fetchall()

Text Cell 4: Section 4.0 – Source Adapter Protocol & Concrete Adapters
Robust API Interrogation
This cell updates the ArxivSourceAdapter:
It inspects the Atom feed's <link> elements to parse the exact PDF link generated by arXiv.
It strips out trailing .pdf from versioned IDs where arXiv returns 404s, ensuring the canonical endpoint https://arxiv.org/pdf/{arxiv_id} is targeted[1].
It sets a standard, non-generic HTTP User-Agent to satisfy open access API gateway policies[2][3].

In [15]:
class BaseSourceAdapter:
    """Abstract Interface defining the Source Adapter contract."""
    source_name: str = "BASE_SOURCE"

    def search(self, query: str, limit: int = 25) -> Generator[CandidateArtifact, None, None]:
        raise NotImplementedError

    def resolve(self, source_record_id: str) -> Optional[CandidateArtifact]:
        raise NotImplementedError


class ArxivSourceAdapter(BaseSourceAdapter):
    """Production Adapter for arXiv.org API (STEM Preprints & Research Papers)."""
    source_name: str = "ARXIV"
    BASE_URL: str = "http://export.arxiv.org/api/query"

    def search(self, query: str, limit: int = 10) -> Generator[CandidateArtifact, None, None]:
        import xml.etree.ElementTree as ET

        params = {
            "search_query": f"all:{query}",
            "start": 0,
            "max_results": limit
        }
        encoded_url = f"{self.BASE_URL}?{urllib.parse.urlencode(params)}"
        req = urllib.request.Request(
            encoded_url,
            headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) MTC3-DualPhD-AcquisitionEngine/2.0"}
        )

        try:
            with urllib.request.urlopen(req, timeout=20) as response:
                xml_data = response.read().decode("utf-8")

            root = ET.fromstring(xml_data)
            namespace = {"atom": "http://www.w3.org/2005/Atom"}

            for entry in root.findall("atom:entry", namespace):
                raw_id = entry.find("atom:id", namespace).text.strip()
                arxiv_id = raw_id.split("/abs/")[-1]
                title = entry.find("atom:title", namespace).text.strip().replace("\n", " ")
                summary = entry.find("atom:summary", namespace).text.strip().replace("\n", " ")

                authors = [a.find("atom:name", namespace).text.strip() for a in entry.findall("atom:author", namespace)]
                primary_author = authors[0] if authors else "Unknown"

                # Extract exact PDF link from atom link attributes
                pdf_url = None
                for link in entry.findall("atom:link", namespace):
                    if link.attrib.get("title") == "pdf" or link.attrib.get("type") == "application/pdf":
                        pdf_url = link.attrib.get("href")
                        break

                # Fallback to canonical arXiv PDF link format
                if not pdf_url:
                    pdf_url = f"https://arxiv.org/pdf/{arxiv_id}"
                elif pdf_url.startswith("http://"):
                    pdf_url = "https://" + pdf_url[7:]

                published = entry.find("atom:published", namespace).text[:4]
                year = int(published) if published.isdigit() else None

                artifact_key = CandidateArtifact.generate_key(title, primary_author, ArtifactType.RESEARCH_PAPER.value)

                yield CandidateArtifact(
                    artifact_key=artifact_key,
                    artifact_type=ArtifactType.RESEARCH_PAPER,
                    canonical_title=title,
                    creators=authors,
                    publication_year=year,
                    language="en",
                    rights_license="open_access",
                    target_format="pdf",
                    discovery_url=raw_id,
                    acquisition_url=pdf_url,
                    source_name=self.source_name,
                    source_id=arxiv_id,
                    metadata={"abstract": summary, "primary_author": primary_author},
                    evidence={"raw_feed": "arxiv_api", "published": published}
                )
        except Exception as e:
            logger.error(f"Error querying arXiv adapter: {e}")


class GutendexSourceAdapter(BaseSourceAdapter):
    """Production Adapter for Project Gutenberg via Gutendex REST API."""
    source_name: str = "GUTENBERG"
    BASE_URL: str = "https://gutendex.com/books"

    def search(self, query: str, limit: int = 10) -> Generator[CandidateArtifact, None, None]:
        params = {"search": query}
        encoded_url = f"{self.BASE_URL}?{urllib.parse.urlencode(params)}"
        req = urllib.request.Request(
            encoded_url,
            headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) MTC3-DualPhD-AcquisitionEngine/2.0"}
        )

        try:
            with urllib.request.urlopen(req, timeout=20) as response:
                data = json.loads(response.read().decode("utf-8"))

            for item in data.get("results", [])[:limit]:
                title = item.get("title", "Untitled").replace("\n", " ")
                authors = [a.get("name", "Unknown") for a in item.get("authors", [])]
                primary_author = authors[0] if authors else "Unknown"
                book_id = str(item.get("id"))

                formats = item.get("formats", {})
                acq_url = (
                    formats.get("application/epub+zip") or
                    formats.get("application/pdf") or
                    formats.get("text/plain; charset=utf-8") or
                    formats.get("text/plain")
                )

                if not acq_url:
                    continue

                target_format = "epub" if "epub" in acq_url else ("pdf" if "pdf" in acq_url else "txt")
                languages = item.get("languages", ["en"])
                primary_lang = languages[0] if languages else "en"

                artifact_key = CandidateArtifact.generate_key(title, primary_author, ArtifactType.BOOK.value)

                yield CandidateArtifact(
                    artifact_key=artifact_key,
                    artifact_type=ArtifactType.BOOK,
                    canonical_title=title,
                    creators=authors,
                    publication_year=None,
                    language=primary_lang,
                    rights_license="public_domain",
                    target_format=target_format,
                    discovery_url=f"https://www.gutenberg.org/ebooks/{book_id}",
                    acquisition_url=acq_url,
                    source_name=self.source_name,
                    source_id=book_id,
                    metadata={"download_count": item.get("download_count", 0), "subjects": item.get("subjects", [])},
                    evidence={"gutenberg_id": book_id}
                )
        except Exception as e:
            logger.error(f"Error querying Gutendex adapter: {e}")

Text Cell 5: Section 5.0 – Declarative Qualification Engine (Policy Gates)
Policy-Driven Decisions
The QualificationEngine separates discovery from pipeline execution. It enforces accept/reject gates, filtering noise or out-of-scope files before network requests are queued for download.

In [16]:
@dataclass
class QualificationPolicy:
    """Declarative Policy schema defining qualification criteria."""
    artifact_type: ArtifactType
    allowed_languages: List[str]
    allowed_licenses: List[str]
    allowed_formats: List[str]
    required_keywords: List[str] = field(default_factory=list)
    banned_keywords: List[str] = field(default_factory=list)


class QualificationDecision(str, Enum):
    QUALIFIED = "QUALIFIED"
    REJECTED = "REJECTED"


class QualificationEngine:
    """Evaluates CandidateArtifacts against declarative profile policies."""

    def __init__(self, policy: QualificationPolicy):
        self.policy = policy

    def evaluate(self, candidate: CandidateArtifact) -> Tuple[QualificationDecision, str]:
        # 1. Validate Artifact Type
        if candidate.artifact_type != self.policy.artifact_type:
            return QualificationDecision.REJECTED, f"Mismatched type: {candidate.artifact_type} != {self.policy.artifact_type}"

        # 2. Language Gate
        if candidate.language not in self.policy.allowed_languages:
            return QualificationDecision.REJECTED, f"Unsupported language: {candidate.language}"

        # 3. Rights / License Gate
        if candidate.rights_license not in self.policy.allowed_licenses:
            return QualificationDecision.REJECTED, f"Unauthorized rights profile: {candidate.rights_license}"

        # 4. Format Gate
        if candidate.target_format not in self.policy.allowed_formats:
            return QualificationDecision.REJECTED, f"Disallowed format: {candidate.target_format}"

        # 5. Negative Keyword Filter (Reject Gates)
        text_corpus = f"{candidate.canonical_title} {json.dumps(candidate.metadata)}".lower()
        for banned in self.policy.banned_keywords:
            if banned.lower() in text_corpus:
                return QualificationDecision.REJECTED, f"Triggered banned keyword: '{banned}'"

        # 6. Positive Keyword Gate (if configured)
        if self.policy.required_keywords:
            matched = any(req.lower() in text_corpus for req in self.policy.required_keywords)
            if not matched:
                return QualificationDecision.REJECTED, "Failed required keywords match"

        return QualificationDecision.QUALIFIED, "Passed all qualification gates."

Text Cell 6: Section 6.0 – Discovery Engine & Canonical Deduplication
Discovery Coordination
The DiscoveryEngine accepts search directives, queries registered adapters, performs canonical deduplication via the Ledger, and routes new items through the QualificationEngine.

In [17]:
class DiscoveryEngine:
    """Coordinates Source Adapters, executes Qualification, and writes state to the Ledger."""

    def __init__(self, ledger: LedgerDB, qualification_engine: QualificationEngine):
        self.ledger = ledger
        self.qualifier = qualification_engine
        self.adapters: Dict[str, BaseSourceAdapter] = {}

    def register_adapter(self, adapter: BaseSourceAdapter):
        self.adapters[adapter.source_name] = adapter
        self.ledger.register_source(
            source_id=adapter.source_name,
            name=adapter.__class__.__name__,
            adapter_cls=adapter.__class__.__name__
        )

    def discover(self, query: str, limit_per_source: int = 5) -> Dict[str, int]:
        stats = {"discovered": 0, "qualified": 0, "skipped": 0, "duplicates": 0}

        for name, adapter in self.adapters.items():
            logger.info(f"Interrogating Source Adapter: [{name}] for query: '{query}'")
            for candidate in adapter.search(query=query, limit=limit_per_source):
                stats["discovered"] += 1

                # 1. Record discovery to Ledger
                is_new, artifact_key = self.ledger.record_discovery(candidate)
                if not is_new:
                    stats["duplicates"] += 1
                    logger.info(f"Duplicate artifact provenance mapped: {candidate.canonical_title[:40]}...")
                    continue

                # 2. Qualify Candidate
                decision, reason = self.qualifier.evaluate(candidate)
                if decision == QualificationDecision.QUALIFIED:
                    stats["qualified"] += 1
                    self.ledger.update_artifact_status(artifact_key, ArtifactStatus.QUEUED)
                    logger.info(f"QUALIFIED -> QUEUED: [{candidate.target_format.upper()}] {candidate.canonical_title[:50]}")
                else:
                    stats["skipped"] += 1
                    self.ledger.update_artifact_status(artifact_key, ArtifactStatus.SKIPPED)
                    logger.debug(f"SKIPPED: {candidate.canonical_title[:40]} | Reason: {reason}")

        return stats

Text Cell 7: Section 7.0 – Hardened Acquisition Engine (Direct Stream & Checksums)
Streamed Binary Acquisition & Verification
The AcquisitionEngine retrieves QUEUED items using their exact acquisition_url. It provides:
Standard browser headers (Accept: application/pdf, User-Agent) to prevent edge gateway 403/404 blocks[2][3].
64KB chunked streaming directly to staging disk.
Simultaneous SHA-256 computation.
Byte size verification (
>
2
 KB
>2 KB
) to verify that empty or rate-limited HTML pages are not treated as valid PDFs.

In [18]:
class AcquisitionEngine:
    """Pulls QUEUED artifacts from the ledger, downloads binaries, verifies integrity, and updates records."""

    def __init__(self, ledger: LedgerDB, staging_dir: Path = STAGING_DIR, max_retries: int = 3):
        self.ledger = ledger
        self.staging_dir = staging_dir
        self.max_retries = max_retries

    def _stream_download(self, url: str, destination: Path) -> Tuple[str, int]:
        """Streams binary data to disk while computing SHA-256 digest in real-time."""
        sha256 = hashlib.sha256()
        total_bytes = 0

        req = urllib.request.Request(
            url,
            headers={
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
                "Accept": "application/pdf,application/epub+zip,application/octet-stream,*/*"
            }
        )

        with urllib.request.urlopen(req, timeout=30) as response, open(destination, "wb") as f:
            while chunk := response.read(64 * 1024):
                sha256.update(chunk)
                f.write(chunk)
                total_bytes += len(chunk)

        return sha256.hexdigest(), total_bytes

    def process_queue(self, batch_size: int = 5) -> int:
        queued_items = self.ledger.get_pending_acquisitions(limit=batch_size)
        acquired_count = 0

        for item in queued_items:
            artifact_key = item["artifact_key"]
            target_format = item["target_format"]
            title = item["canonical_title"]
            acq_url = item["acquisition_url"]

            staging_file = self.staging_dir / f"{artifact_key}.{target_format}"
            logger.info(f"Acquiring [{artifact_key[:8]}]: {title[:40]} via {acq_url}")

            success = False
            for attempt in range(1, self.max_retries + 1):
                try:
                    time.sleep(1.5)  # Respect upstream rate limits
                    checksum, byte_size = self._stream_download(acq_url, staging_file)

                    # Sanity check: Ensure payload is not an empty error stub
                    if byte_size < 2048:
                        raise ValueError(f"Acquired payload suspicious: byte size {byte_size} < 2KB")

                    self.ledger.record_acquisition(
                        artifact_key=artifact_key,
                        url=acq_url,
                        status=AcquisitionStatus.VERIFIED,
                        local_path=str(staging_file),
                        checksum=checksum,
                        byte_size=byte_size
                    )
                    self.ledger.update_artifact_status(artifact_key, ArtifactStatus.ACQUIRED)
                    logger.info(f"ACQUIRED: {title[:40]} | Size: {byte_size / 1024:.1f} KB | Hash: {checksum[:8]}...")
                    acquired_count += 1
                    success = True
                    break

                except Exception as e:
                    logger.warning(f"Acquisition Attempt {attempt}/{self.max_retries} failed for {artifact_key[:8]}: {e}")
                    if staging_file.exists():
                        staging_file.unlink()
                    time.sleep(2.0 * attempt)

            if not success:
                logger.error(f"PERMANENT FAILURE: Unable to acquire artifact {artifact_key}")
                self.ledger.record_acquisition(
                    artifact_key=artifact_key,
                    url=acq_url,
                    status=AcquisitionStatus.FAILED,
                    error_msg="Max retries exhausted"
                )
                self.ledger.update_artifact_status(artifact_key, ArtifactStatus.FAILED)

        return acquired_count

Text Cell 8: Section 8.0 – Idempotent Delivery Engine
Packaging & Companion Metadata Export
The DeliveryEngine transfers verified assets from staging to delivery, creating structured subdirectories by artifact type and generating companion JSON metadata sidecar files for downstream analysis.

In [19]:
class DeliveryEngine:
    """Transfers ACQUIRED artifacts to persistent destination layouts and exports companion metadata."""

    def __init__(self, ledger: LedgerDB, delivery_base: Path = DELIVERY_DIR):
        self.ledger = ledger
        self.delivery_base = delivery_base

    def deliver_acquired(self) -> int:
        with self.ledger._get_connection() as conn:
            cursor = conn.execute("""
            SELECT a.artifact_key, a.artifact_type, a.canonical_title, a.target_format, a.metadata_json,
                   ac.local_path, ac.checksum, ac.byte_size
            FROM artifacts a
            JOIN acquisitions ac ON a.artifact_key = ac.artifact_key
            WHERE a.status = ? AND ac.status = ?
            """, (ArtifactStatus.ACQUIRED.value, AcquisitionStatus.VERIFIED.value))
            ready_items = cursor.fetchall()

        delivered_count = 0
        for item in ready_items:
            key = item["artifact_key"]
            atype = item["artifact_type"]
            src_path = Path(item["local_path"])

            if not src_path.exists():
                logger.error(f"Delivery failed: Source binary missing at {src_path}")
                continue

            dest_dir = self.delivery_base / atype
            dest_dir.mkdir(parents=True, exist_ok=True)

            dest_binary = dest_dir / f"{key}.{item['target_format']}"
            dest_meta = dest_dir / f"{key}.meta.json"

            # Transfer payload
            with open(src_path, "rb") as f_src, open(dest_binary, "wb") as f_dst:
                f_dst.write(f_src.read())

            # Write companion JSON metadata sidecar
            metadata_payload = {
                "artifact_key": key,
                "title": item["canonical_title"],
                "artifact_type": atype,
                "checksum_sha256": item["checksum"],
                "byte_size": item["byte_size"],
                "delivered_at": get_utc_now(),
                "metadata": json.loads(item["metadata_json"]) if item["metadata_json"] else {}
            }
            with open(dest_meta, "w", encoding="utf-8") as f:
                json.dump(metadata_payload, f, indent=2)

            # Record state
            self.ledger.record_delivery(
                artifact_key=key,
                dest_type="LOCAL_STRUCTURED_STORE",
                dest_path=str(dest_binary)
            )
            self.ledger.update_artifact_status(key, ArtifactStatus.DELIVERED)

            # Clean staging
            src_path.unlink(missing_ok=True)
            delivered_count += 1
            logger.info(f"DELIVERED -> [{dest_binary.name}]")

        return delivered_count

Text Cell 9: Section 9.0 – The Hunt Controller (Intent-Driven Search Planning)
Dynamic Search Planning
The HuntController computes the mathematical delta between target acquisition intent and current ledger inventory, deciding dynamically when search quotas are fulfilled.


In [20]:
@dataclass
class HuntIntent:
    target_profile: str                   # Profile identifier (e.g., 'DUAL_PHD_MATH_CS')
    target_count: int                     # Target number of artifacts
    search_terms: List[str]               # Ordered keyword exploration list
    artifact_type: ArtifactType           # BOOK, RESEARCH_PAPER, etc.


class HuntController:
    """Evaluates ledger coverage against high-level intent and issues directed search directives."""

    def __init__(self, ledger: LedgerDB, intent: HuntIntent):
        self.ledger = ledger
        self.intent = intent

    def get_coverage_delta(self) -> int:
        with self.ledger._get_connection() as conn:
            cursor = conn.execute("""
            SELECT COUNT(*) FROM artifacts
            WHERE artifact_type = ? AND status IN (?, ?)
            """, (self.intent.artifact_type.value, ArtifactStatus.DELIVERED.value, ArtifactStatus.QUEUED.value))
            current_count = cursor.fetchone()[0]

        return max(0, self.intent.target_count - current_count)

    def plan_next_search_vector(self) -> List[str]:
        delta = self.get_coverage_delta()
        if delta <= 0:
            logger.info("Goal reached: No further exploration vectors required.")
            return []

        logger.info(f"Hunt Gap Analysis: {delta} items required to satisfy target.")
        return self.intent.search_terms

Text Cell 10: Section 10.0 – Central Orchestrator & State Machine Controller
Pipeline Execution Loop
The PipelineOrchestrator schedules the sub-engines and prints an executive audit report of the durable ledger.

In [21]:
class PipelineOrchestrator:
    """Master controller scheduling, invoking, and auditing the full Hunt-Acquire-Deliver lifecycle."""

    def __init__(self,
                 ledger: LedgerDB,
                 discovery: DiscoveryEngine,
                 acquisition: AcquisitionEngine,
                 delivery: DeliveryEngine,
                 hunt_controller: HuntController):
        self.ledger = ledger
        self.discovery = discovery
        self.acquisition = acquisition
        self.delivery = delivery
        self.hunt_controller = hunt_controller

    def execute_hunt_cycle(self, limit_per_vector: int = 2):
        run_id = f"RUN_{int(time.time())}"
        logger.info(f"=== INITIATING ACQUISITION CYCLE [{run_id}] ===")

        # 1. Evaluate Intent Plan
        search_vectors = self.hunt_controller.plan_next_search_vector()
        if not search_vectors:
            logger.info("Pipeline idle: Hunt goals satisfied.")
            return

        # 2. Discovery Phase
        discovery_summary = {"discovered": 0, "qualified": 0, "skipped": 0, "duplicates": 0}
        for query in search_vectors:
            stats = self.discovery.discover(query=query, limit_per_source=limit_per_vector)
            for k in discovery_summary:
                discovery_summary[k] += stats[k]

        logger.info(f"Discovery Phase Completed: {discovery_summary}")

        # 3. Acquisition Phase
        logger.info("Triggering Acquisition Engine Queue...")
        acquired_total = self.acquisition.process_queue(batch_size=10)
        logger.info(f"Acquisition Phase Completed: {acquired_total} items verified.")

        # 4. Delivery Phase
        logger.info("Triggering Delivery Engine...")
        delivered_total = self.delivery.deliver_acquired()
        logger.info(f"Delivery Phase Completed: {delivered_total} items archived.")

        logger.info(f"=== COMPLETED ACQUISITION CYCLE [{run_id}] ===")

    def generate_audit_report(self):
        """Prints a comprehensive tabular audit of the persistent ledger state."""
        with self.ledger._get_connection() as conn:
            cursor = conn.execute("""
            SELECT status, COUNT(*) as count FROM artifacts GROUP BY status
            """)
            status_counts = dict(cursor.fetchall())

            cursor = conn.execute("""
            SELECT a.artifact_key, a.canonical_title, a.status, a.target_format,
                   ac.byte_size, ac.checksum, d.source_id
            FROM artifacts a
            LEFT JOIN acquisitions ac ON a.artifact_key = ac.artifact_key AND ac.status = 'VERIFIED'
            LEFT JOIN discoveries d ON a.artifact_key = d.artifact_key
            GROUP BY a.artifact_key
            """)
            rows = cursor.fetchall()

        print("\n" + "="*85)
        print("                  MTC3 ACQUISITION LEDGER AUDIT REPORT")
        print("="*85)
        print(f"Artifact Status Summary: {json.dumps(status_counts, indent=2)}")
        print("-"*85)
        print(f"{'KEY':<10} | {'STATUS':<11} | {'FMT':<4} | {'SIZE (KB)':<10} | {'TITLE'}")
        print("-"*85)
        for r in rows:
            size_kb = f"{r['byte_size']/1024:.1f}" if r['byte_size'] else "N/A"
            title = r['canonical_title'][:42] + "..." if len(r['canonical_title']) > 42 else r['canonical_title']
            print(f"{r['artifact_key'][:10]:<10} | {r['status']:<11} | {r['target_format']:<4} | {size_kb:<10} | {title}")
        print("="*85 + "\n")

Text Cell 11: Section 11.0 – Profile 001 Live Execution Run & Audit Verification
Dual PhD Applied Mathematics / CS Target Run
This final cell executes a fresh run targeting real STEM research papers from arXiv:
Resets any previous SQLite database locks.
Runs the pipeline across optimization and numerical consensus queries.
Verifies that binaries are streamed, checksummed, delivered to the file tree, and recorded with DELIVERED status.


In [22]:
# 1. Clean previous runs if testing fresh
if LEDGER_PATH.exists():
    LEDGER_PATH.unlink()

# 2. Instantiate Ledger
ledger = LedgerDB(LEDGER_PATH)

# 3. Define Declarative Qualification Policy
stem_policy = QualificationPolicy(
    artifact_type=ArtifactType.RESEARCH_PAPER,
    allowed_languages=["en"],
    allowed_licenses=["open_access", "public_domain"],
    allowed_formats=["pdf"],
    required_keywords=[],
    banned_keywords=["cryptocurrency", "nft"]
)
qualification_engine = QualificationEngine(stem_policy)

# 4. Instantiate Sub-Engines
discovery_engine = DiscoveryEngine(ledger, qualification_engine)
discovery_engine.register_adapter(ArxivSourceAdapter())

acquisition_engine = AcquisitionEngine(ledger, staging_dir=STAGING_DIR)
delivery_engine = DeliveryEngine(ledger, delivery_base=DELIVERY_DIR)

# 5. Define Hunt Intent
phd_intent = HuntIntent(
    target_profile="DUAL_PHD_MATH_CS",
    target_count=2,
    search_terms=[
        "convex optimization algorithms",
        "distributed consensus numerical methods"
    ],
    artifact_type=ArtifactType.RESEARCH_PAPER
)
hunt_controller = HuntController(ledger, phd_intent)

# 6. Build and Execute Orchestrator
orchestrator = PipelineOrchestrator(
    ledger=ledger,
    discovery=discovery_engine,
    acquisition=acquisition_engine,
    delivery=delivery_engine,
    hunt_controller=hunt_controller
)

# Execute cycle (pulling 2 papers per query term)
orchestrator.execute_hunt_cycle(limit_per_vector=1)

# 7. Print System-of-Record Audit Report
orchestrator.generate_audit_report()

# 8. Verify Physical Deliverables
print("Delivered Physical Artifact Directory Hierarchy:")
for root, dirs, files in os.walk(DELIVERY_DIR):
    level = root.replace(str(DELIVERY_DIR), '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")


                  MTC3 ACQUISITION LEDGER AUDIT REPORT
Artifact Status Summary: {
  "DELIVERED": 2
}
-------------------------------------------------------------------------------------
KEY        | STATUS      | FMT  | SIZE (KB)  | TITLE
-------------------------------------------------------------------------------------
98f837a099 | DELIVERED   | pdf  | 1595.1     | A Flux Conserving Meshfree Method for Cons...
9e33893707 | DELIVERED   | pdf  | 1120.2     | Gradient-based Algorithms for Convex Discr...

Delivered Physical Artifact Directory Hierarchy:
delivery/
    RESEARCH_PAPER/
        98f837a09987dc3d75a7.meta.json
        9e338937077a00ee17c7.meta.json
        9e338937077a00ee17c7.pdf
        98f837a09987dc3d75a7.pdf


Text Cell 12: Section 12.0 – Manifest Domain Model & Precision Title Matcher
Architectural Shift: Exploratory Search
→
→
 Deterministic Manifest Hunting
Exploratory Hunting: "Find any 5 open-access papers about Convex Optimization."
Targeted Manifest Hunting: "Find these exact 6 books by title and author; verify their identity; acquire and map each to the manifest."
Core Mechanics Added:
ManifestItem: Represents an individual target (title, author, optional ISBN/ID, and format priority).
Token & Sequence Matcher: Uses standard-library normalized string distance (difflib.SequenceMatcher) to prevent false positives when querying open repositories with generic search results.

In [23]:
import difflib

@dataclass
class ManifestItem:
    """Represents a specific target book or text in a desired reading list."""
    title: str
    author: str = ""
    identifier: Optional[str] = None      # ISBN, Gutenberg ID, DOI, or ArXiv ID
    min_similarity_score: float = 0.65    # Required match threshold (0.0 to 1.0)
    preferred_formats: List[str] = field(default_factory=lambda: ["epub", "pdf", "txt"])

    def get_search_query(self) -> str:
        """Constructs an optimal query string for source adapters."""
        if self.identifier:
            return self.identifier
        if self.author:
            return f"{self.title} {self.author}"
        return self.title


class PrecisionMatcher:
    """Normalizes and computes semantic text similarity between candidate metadata and manifest targets."""

    @staticmethod
    def normalize_text(text: str) -> str:
        return "".join(ch.lower() for ch in text if ch.isalnum() or ch.isspace()).strip()

    @classmethod
    def evaluate_match(cls, manifest_item: ManifestItem, candidate: CandidateArtifact) -> Tuple[bool, float, str]:
        # 1. Direct ID match (instant verification)
        if manifest_item.identifier and manifest_item.identifier.lower() in candidate.source_id.lower():
            return True, 1.0, f"Exact ID Match: {manifest_item.identifier}"

        norm_target_title = cls.normalize_text(manifest_item.title)
        norm_cand_title = cls.normalize_text(candidate.canonical_title)

        # 2. Compute Title Similarity (SequenceMatcher ratio)
        title_score = difflib.SequenceMatcher(None, norm_target_title, norm_cand_title).ratio()

        # Check substring containment (e.g. "The Republic" in "The Republic of Plato")
        if norm_target_title in norm_cand_title or norm_cand_title in norm_target_title:
            title_score = max(title_score, 0.85)

        # 3. Compute Author Similarity if specified
        author_score = 1.0
        if manifest_item.author:
            norm_target_author = cls.normalize_text(manifest_item.author)
            cand_authors = " ".join([cls.normalize_text(a) for a in candidate.creators])

            author_matcher = difflib.SequenceMatcher(None, norm_target_author, cand_authors)
            author_score = author_matcher.ratio()

            # Substring match for author last name
            for part in norm_target_author.split():
                if len(part) > 3 and part in cand_authors:
                    author_score = max(author_score, 0.80)

        # 4. Composite Confidence Score
        composite_score = (title_score * 0.70) + (author_score * 0.30)
        is_match = composite_score >= manifest_item.min_similarity_score

        reason = f"Title Score: {title_score:.2f}, Author Score: {author_score:.2f} (Composite: {composite_score:.2f})"
        return is_match, composite_score, reason

Text Cell 13: Section 13.0 – Manifest-Aware Hunt Controller & Precision Qualification
The Manifest Hunt Controller
The ManifestHuntController audits the durable Ledger before querying any external source:
It compares the user's ManifestItem list against existing DELIVERED and ACQUIRED artifacts.
If an item is already safely stored on disk, it is skipped immediately (Zero redundant bandwidth).
If an item is unfulfilled, it schedules targeted search vectors across all active source adapters.

In [24]:
class ManifestHuntController:
    """Manages the lifecycle of an explicit target manifest list."""

    def __init__(self, ledger: LedgerDB, manifest: List[ManifestItem]):
        self.ledger = ledger
        self.manifest = manifest

    def get_unfulfilled_targets(self) -> List[ManifestItem]:
        """Returns only items from the manifest that are not yet acquired or delivered."""
        unfulfilled = []
        with self.ledger._get_connection() as conn:
            for item in self.manifest:
                norm_title = f"%{PrecisionMatcher.normalize_text(item.title)[:20]}%"
                cursor = conn.execute("""
                SELECT COUNT(*) FROM artifacts
                WHERE LOWER(canonical_title) LIKE ? AND status IN (?, ?)
                """, (norm_title.lower(), ArtifactStatus.DELIVERED.value, ArtifactStatus.ACQUIRED.value))
                count = cursor.fetchone()[0]
                if count == 0:
                    unfulfilled.append(item)
        return unfulfilled


class TargetedDiscoveryEngine(DiscoveryEngine):
    """Specialized Discovery Engine that matches search candidates strictly against manifest items."""

    def hunt_manifest(self, manifest_items: List[ManifestItem], limit_per_source: int = 5) -> Dict[str, Any]:
        stats = {"searched_targets": len(manifest_items), "matched_and_queued": 0, "rejected_candidates": 0}

        for target in manifest_items:
            query = target.get_search_query()
            logger.info(f"Targeted Hunt: '{target.title}' by '{target.author}' (Query: '{query}')")

            target_satisfied = False
            for adapter_name, adapter in self.adapters.items():
                if target_satisfied:
                    break

                for candidate in adapter.search(query=query, limit=limit_per_source):
                    # 1. Check Precision Similarity Match against manifest
                    is_match, score, match_reason = PrecisionMatcher.evaluate_match(target, candidate)

                    if not is_match:
                        stats["rejected_candidates"] += 1
                        logger.debug(f"Rejecting candidate: '{candidate.canonical_title}' - {match_reason}")
                        continue

                    # 2. Record to Ledger
                    is_new, artifact_key = self.ledger.record_discovery(candidate)

                    # 3. Qualify Candidate
                    decision, qual_reason = self.qualifier.evaluate(candidate)
                    if decision == QualificationDecision.QUALIFIED:
                        self.ledger.update_artifact_status(artifact_key, ArtifactStatus.QUEUED)
                        logger.info(f"TARGET MATCH QUALIFIED -> QUEUED: '{candidate.canonical_title}' ({match_reason})")
                        stats["matched_and_queued"] += 1
                        target_satisfied = True
                        break
                    else:
                        self.ledger.update_artifact_status(artifact_key, ArtifactStatus.SKIPPED)
                        logger.info(f"Target matched but failed policy: {candidate.canonical_title} ({qual_reason})")

        return stats

Text Cell 14: Section 14.0 – Manifest Reconciliation & Final Audit Reporting
Verification & Reconciliation Loop
When running against a specific list, an acquisition system must report Reconciliation Telemetry:
Which target books were successfully found, downloaded, and verified?
Which specific books remain unfulfilled (missing from open access / public domain endpoints)?

In [25]:
def generate_manifest_reconciliation_report(ledger: LedgerDB, manifest: List[ManifestItem]):
    """Reconciles user manifest items against persistent ledger records."""
    print("\n" + "="*95)
    print("                    MTC3 TARGET MANIFEST RECONCILIATION REPORT")
    print("="*95)
    print(f"{'TARGET TITLE':<35} | {'TARGET AUTHOR':<18} | {'STATUS':<12} | {'DELIVERED FILE / KEY'}")
    print("-"*95)

    with ledger._get_connection() as conn:
        for item in manifest:
            norm_title = f"%{PrecisionMatcher.normalize_text(item.title)[:15]}%"
            cursor = conn.execute("""
            SELECT a.artifact_key, a.status, a.target_format, d.destination_path
            FROM artifacts a
            LEFT JOIN deliveries d ON a.artifact_key = d.artifact_key
            WHERE LOWER(a.canonical_title) LIKE ?
            ORDER BY a.created_at DESC LIMIT 1
            """, (norm_title.lower(),))
            row = cursor.fetchone()

            t_title = item.title[:33] + ".." if len(item.title) > 33 else item.title
            t_author = item.author[:16] if item.author else "N/A"

            if row:
                status = row["status"]
                file_info = Path(row["destination_path"]).name if row["destination_path"] else row["artifact_key"][:12]
            else:
                status = "NOT_FOUND"
                file_info = "Pending upstream discovery"

            print(f"{t_title:<35} | {t_author:<18} | {status:<12} | {file_info}")

    print("="*95 + "\n")

Text Cell 15: Section 15.0 – Live Manifest Hunt Execution (Reading List Profile)
Executing a Real Book & Manuscript Manifest
In this cell, we supply an explicit list of foundational texts across mathematics, computing, and classical science (e.g., Fourier, Russell, Euler, Euclid, Boole, and modern preprints).
The engine searches Project Gutenberg (Gutendex) and arXiv, filters out irrelevant results via PrecisionMatcher, downloads valid payloads, delivers companion .meta.json records, and outputs the final manifest reconciliation table.

In [26]:
# 1. Define your explicit Target Book/Paper Manifest
target_manifest = [
    ManifestItem(
        title="An Analytical Theory of Heat",
        author="Joseph Fourier",
        preferred_formats=["epub", "pdf", "txt"]
    ),
    ManifestItem(
        title="An Investigation of the Laws of Thought",
        author="George Boole",
        preferred_formats=["epub", "pdf", "txt"]
    ),
    ManifestItem(
        title="The Problems of Philosophy",
        author="Bertrand Russell",
        preferred_formats=["epub", "pdf"]
    ),
    ManifestItem(
        title="Convex Optimization: Algorithms and Complexity",
        author="Sebastien Bubeck",
        preferred_formats=["pdf"]
    ),
    ManifestItem(
        title="Relativity: The Special and General Theory",
        author="Albert Einstein",
        preferred_formats=["epub", "pdf"]
    )
]

# 2. Setup Qualification Policy supporting both BOOK and RESEARCH_PAPER
book_policy = QualificationPolicy(
    artifact_type=ArtifactType.BOOK,
    allowed_languages=["en"],
    allowed_licenses=["public_domain", "open_access"],
    allowed_formats=["epub", "pdf", "txt"]
)
book_qualifier = QualificationEngine(book_policy)

# 3. Instantiate Ledger & Targeted Discovery Engine
ledger = LedgerDB(LEDGER_PATH)
targeted_discovery = TargetedDiscoveryEngine(ledger, book_qualifier)

# Register Adapters for both Public Domain Books and STEM Research Papers
gutendex_adapter = GutendexSourceAdapter()
arxiv_adapter = ArxivSourceAdapter()

targeted_discovery.register_adapter(gutendex_adapter)
targeted_discovery.register_adapter(arxiv_adapter)

acquisition_engine = AcquisitionEngine(ledger, staging_dir=STAGING_DIR)
delivery_engine = DeliveryEngine(ledger, delivery_base=DELIVERY_DIR)
manifest_controller = ManifestHuntController(ledger, target_manifest)

# 4. Check for Unfulfilled Manifest Items
pending_targets = manifest_controller.get_unfulfilled_targets()
logger.info(f"Targeted Manifest Initialized: {len(pending_targets)} of {len(target_manifest)} books needed.")

# 5. Execute Targeted Discovery
discovery_stats = targeted_discovery.hunt_manifest(pending_targets, limit_per_source=3)
logger.info(f"Discovery Results: {discovery_stats}")

# 6. Process Binary Downloads & Verification
logger.info("Executing Acquisition Engine for Qualified Targets...")
acquired_count = acquisition_engine.process_queue(batch_size=10)

# 7. Deliver & Stage Verified Deliverables
logger.info("Executing Delivery Engine...")
delivered_count = delivery_engine.deliver_acquired()

# 8. Print Executive Manifest Reconciliation Table
generate_manifest_reconciliation_report(ledger, target_manifest)

# 9. Verify Final Delivery Workspace
print("Delivered Physical Artifact Directory Hierarchy:")
for root, dirs, files in os.walk(DELIVERY_DIR):
    level = root.replace(str(DELIVERY_DIR), '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")

ERROR:AcquisitionEngine:Error querying Gutendex adapter: The read operation timed out



                    MTC3 TARGET MANIFEST RECONCILIATION REPORT
TARGET TITLE                        | TARGET AUTHOR      | STATUS       | DELIVERED FILE / KEY
-----------------------------------------------------------------------------------------------
An Analytical Theory of Heat        | Joseph Fourier     | NOT_FOUND    | Pending upstream discovery
An Investigation of the Laws of T.. | George Boole       | NOT_FOUND    | Pending upstream discovery
The Problems of Philosophy          | Bertrand Russell   | DELIVERED    | 13d2fdaf26fdc34ece4b.epub
Convex Optimization: Algorithms a.. | Sebastien Bubeck   | SKIPPED      | 9cc1cce31a82
Relativity: The Special and Gener.. | Albert Einstein    | NOT_FOUND    | Pending upstream discovery

Delivered Physical Artifact Directory Hierarchy:
delivery/
    BOOK/
        86da8efc001de7a2e6af.epub
        13d2fdaf26fdc34ece4b.epub
        13d2fdaf26fdc34ece4b.meta.json
        86da8efc001de7a2e6af.meta.json
    RESEARCH_PAPER/
        98f837a0998